# 04 · Final dataset

The Wikipedia polls of all five elections, restricted to the polls kept by thinning (notebook 02), with the
hand-filled validation applied:

1. **Cross-check** (2002, 2022): where the other source is right, its values replace Wikipedia's for the
   candidates that disagree.
2. **Spot check**: one-off errors only (wrong shares, candidates listed but not tested, a few dates).
   Systematic gaps are not corrected, by decision: the 2007 and 2012 dates (publication dates), the missing
   small candidates, the Green candidate's name before nomination. Sample size stays the base of
   respondents.

Every correction, applied or not, is in `data/final/corrections.csv` with its reason. The rules are in
`src/polls/final.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from polls import final

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
OUT = ROOT / "data" / "final"
OUT.mkdir(parents=True, exist_ok=True)

polls, log, warnings = final.build(ROOT)
polls.to_csv(OUT / "polls.csv", index=False)
log.to_csv(OUT / "corrections.csv", index=False)
print(f"{polls['poll_id'].nunique()} polls, {polls['scenario_id'].nunique()} scenarios, {len(polls)} rows")
print("warnings:", *(warnings or ["none"]), sep="\n  ")

708 polls, 1164 scenarios, 16445 rows
warnings:
  none


## Polls per election, and how they were checked

In [2]:
p = polls.drop_duplicates("poll_id")
pd.crosstab(p["election"], p["checked_against"].replace("", "not checked"), margins=True)

checked_against,not checked,original document,other source,other source; original document,All
election,,,,,
2002,1,0,67,30,98
2007,77,30,0,0,107
2012,94,30,0,0,124
2017,120,28,0,0,148
2022,13,2,191,25,231
All,305,90,258,55,708


## Corrections

Applied, by source and field; then what was not applied, and why.

In [3]:
log.pivot_table(index=["source", "field"], columns="applied", values="item", aggfunc="count", fill_value=0)

applied                False  True 
source      field                  
cross-check               29      0
            share          0     95
spot check               128      0
            fieldwork      0      7
            share         20     32

In [4]:
log[~log["applied"]].groupby(["source", "reason"]).size().to_frame("items")

items
source      reason                                                               
cross-check Wikipedia kept                                                     22
            no document: Wikipedia kept                                         6
            not added: the other source's scenario has only 1 candidate(s)      1
spot check  2007/2012 dates are publication dates: kept (decision)             19
            already this value                                                 20
            data keeps the base of respondents (decision)                      25
            dates_ok is not 'no': information only                             15
            information only                                                   36
            missing candidates are not added (decision)                        25
            partial document: absence not conclusive                            5
            systematic (Green candidate before nomination): not corrected       3

**Held** items wait for a check by hand (see `HELD` in `src/polls/final.py`).

In [5]:
log[log["reason"].str.startswith("held")][["poll_id", "item", "reason"]]

,poll_id,item,reason


## Share totals

Shares of the tested candidates by scenario. Totals below 98 in 2007 and 2012 come mostly from the small
candidates Wikipedia leaves out.

In [6]:
v = polls[polls["share_status"] == "value"]
tot = v.groupby(["election", "scenario_id"])["share"].sum().reset_index()
tot.groupby("election")["share"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(1)

,count,min,25%,50%,75%,max
election,,,,,,
2002,98.0,97.0,100.0,100.0,100.0,100.5
2007,109.0,96.0,99.0,100.0,100.0,110.0
2012,166.0,94.5,99.0,100.0,100.0,101.0
2017,344.0,97.5,100.0,100.0,100.0,101.0
2022,447.0,98.0,100.0,100.0,100.0,100.5


In [7]:
print("scenarios summing outside 98-102:")
tot[(tot["share"] < 98) | (tot["share"] > 102)].groupby("election").size()

scenarios summing outside 98-102:


election
2002     2
2007     3
2012    18
2017     1
dtype: int64